In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gs;
CREATE VOLUME IF NOT EXISTS workspace.gs.chegada;

In [0]:
for pasta in ["arvores", "clima", "denuncias", "_controle"]:
    dbutils.fs.mkdirs(f"/Volumes/workspace/gs/chegada/{pasta}")

display(dbutils.fs.ls("/Volumes/workspace/gs/chegada/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/gs/chegada/_controle/,_controle/,0,1794926240761
dbfs:/Volumes/workspace/gs/chegada/arvores/,arvores/,0,1794926240761
dbfs:/Volumes/workspace/gs/chegada/clima/,clima/,0,1794926240761
dbfs:/Volumes/workspace/gs/chegada/denuncias/,denuncias/,0,1794926240761


In [0]:
for f in ["arvores", "clima", "denuncias"]:
    print(f, sorted(x.name for x in dbutils.fs.ls(f"/Volumes/workspace/gs/chegada/{f}")))

arvores ['arvores_2026-11-10.json', 'arvores_2026-11-11.json', 'arvores_2026-11-12.json', 'arvores_2026-11-13.json', 'arvores_2026-11-14.json', 'arvores_2026-11-15.json', 'arvores_2026-11-16.json', 'arvores_2026-11-17.json']
clima ['clima_2026-11-10.json', 'clima_2026-11-11.json', 'clima_2026-11-12.json', 'clima_2026-11-13.json', 'clima_2026-11-14.json', 'clima_2026-11-15.json', 'clima_2026-11-16.json', 'clima_2026-11-17.json']
denuncias ['denuncias_2026-11-10.json', 'denuncias_2026-11-11.json', 'denuncias_2026-11-12.json', 'denuncias_2026-11-13.json', 'denuncias_2026-11-14.json', 'denuncias_2026-11-15.json', 'denuncias_2026-11-16.json', 'denuncias_2026-11-17.json']


### Uploads no Volume por dia (Requisito 04: pelo menos 2 lotes em dias diferentes)
`modificationTime` de cada arquivo mostra quando ele chegou ao Volume. Print desta tabela para o relatório.

In [ ]:
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F

FUSO_SP = timezone(timedelta(hours=-3))
linhas = []
for fonte in ["arvores", "clima", "denuncias"]:
    for f in dbutils.fs.ls(f"/Volumes/workspace/gs/chegada/{fonte}"):
        enviado = datetime.fromtimestamp(f.modificationTime / 1000, FUSO_SP)
        linhas.append((enviado.date().isoformat(), fonte, f.name, enviado.isoformat(timespec="minutes")))

uploads = spark.createDataFrame(linhas, "dia_upload STRING, fonte STRING, arquivo STRING, enviado_em STRING")
display(
    uploads.groupBy("dia_upload")
    .agg(F.count("*").alias("arquivos"), F.array_sort(F.collect_set("arquivo")).alias("arquivos_enviados"))
    .orderBy("dia_upload")
)
dias = uploads.select("dia_upload").distinct().count()
print(f"Lotes de upload em dias diferentes: {dias}" + ("" if dias >= 2 else "  <-- falta o 2o lote (Requisito 04)"))